In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
from concept_abstraction.training import train_ppo_model, evaluate_model, RandomAgent, train_two_stage_ppo_model
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv

import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
import scipy

In [7]:
is_jupyter = 'ipykernel' in sys.modules

In [8]:
if is_jupyter: 
    # Basics 
    seed        = 42
    environment_string = "cyclic_4"
    training_timesteps = 10
    num_concepts_selected = 2
    selection_function = "q_value"
    # Experiment #3
    cbm_accuracy_by_concept = [0.5,0.5,0.5,0.5]  
    cbm_std_by_concept = None 
    target_abstraction = 0.05
    reward_error = 0
    # Experiment #4
    concept_source = "human_selected"
    # Experiment #5
    assess_completeness=False
    out_folder = "cart_pole"
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
    parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
    parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
    parser.add_argument('--cbm_std_by_concept', help="What is the error of AI per concept?", nargs='*', type=float, default=None)
    parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
    parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
    parser.add_argument('--concept_source', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
    parser.add_argument('--assess_completeness', help='Compare to the concept completeness algorithm?', action='store_true')
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    environment_string = args.environment_string
    training_timesteps = args.training_timesteps 
    selection_function = args.selection_function
    num_concepts_selected = args.num_concepts_selected
    cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
    target_abstraction = args.target_abstraction
    reward_error = args.reward_error
    concept_source = args.concept_source
    assess_completeness = args.assess_completeness
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [9]:
results = {}
results['parameters'] = {'seed'      : seed,
        'environment_string'    : environment_string, 
        'training_timesteps': training_timesteps, 
        'selection_function': selection_function,
        'num_concepts_selected': num_concepts_selected,
        'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
        'cbm_std_by_concept': cbm_std_by_concept,
        'target_abstraction': target_abstraction,
        'reward_error': reward_error, 
        'concept_source': concept_source,
        'assess_completeness': assess_completeness
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'cyclic_4', 'training_timesteps': 10, 'selection_function': 'q_value', 'num_concepts_selected': 2, 'cbm_accuracy_by_concept': [0.5, 0.5, 0.5, 0.5], 'cbm_std_by_concept': None, 'target_abstraction': 0.05, 'reward_error': 0, 'concept_source': 'human_selected', 'assess_completeness': False}


In [10]:
np.random.seed(seed)
random.seed(seed)

### Experiment #1&2

In [11]:
concept_list = get_concepts(environment_string,concept_source,seed)

In [12]:
ground_truth_env, ground_truth_eval_env, additional_info = get_environment(environment_string,None)

In [13]:
# Train the groundtruth policy
if "cyclic" in environment_string or "tree" in environment_string or "mimic" in environment_string:
    policy = "MlpPolicy"
else:
    policy = "CnnPolicy"
groundtruth_model = train_ppo_model(ground_truth_env,total_timesteps=training_timesteps,policy=policy)
groundtruth_reward = evaluate_model(environment_string,ground_truth_eval_env,additional_info,groundtruth_model,seed)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [14]:
# Train a random policy
model = RandomAgent(ground_truth_env)
random_reward = evaluate_model(environment_string,ground_truth_eval_env,additional_info,model,seed)

In [15]:
# Train a random selector
subset_concept = random_selection(concept_list,num_concepts_selected)
env, eval_env, additional_info = get_environment(environment_string,subset_concept)
model = train_ppo_model(env,total_timesteps=training_timesteps,policy="MlpPolicy")
random_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)

In [34]:
# Train a greedy selector
subset_concept, greedy_idx = greedy_selection(ground_truth_eval_env,concept_list,num_concepts_selected,groundtruth_model,selection_function)
env, eval_env, additional_info = get_environment(environment_string,subset_concept)
model = train_ppo_model(env,total_timesteps=training_timesteps,policy="MlpPolicy")
greedy_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [48]:
subset_concept, greedy_iterative_idx = greedy_iterative_selection(ground_truth_eval_env,concept_list,num_concepts_selected,groundtruth_model,selection_function)
env, eval_env, additional_info = get_environment(environment_string,subset_concept)
model = train_ppo_model(env,total_timesteps=training_timesteps,policy="MlpPolicy")
greedy_iterative_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [56]:
subset_concept, lp_idx = lp_based_selection(ground_truth_eval_env,concept_list,num_concepts_selected,groundtruth_model,selection_function,target_abstraction)
env, eval_env, additional_info = get_environment(environment_string,subset_concept)
model = train_ppo_model(env,total_timesteps=training_timesteps,policy="MlpPolicy")
lp_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)

74


### Experiment 3

In [128]:
greedy_inaccurate_reward = {}
for modification in ["continuous","binary"]:
    if modification == "continuous" and cbm_std_by_concept is not None:
        modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
    elif modification == "binary" and cbm_accuracy_by_concept is not None:
        modified_concept_predictors = [inaccurate_concepts_binary(func,acc) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
    else:
        continue 
    subset_concept = [modified_concept_predictors[i] for i in greedy_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept)
    model = train_ppo_model(env,total_timesteps=10_000,policy="MlpPolicy")
    greedy_inaccurate_reward[modification] = evaluate_model(environment_string,eval_env,additional_info,model,seed)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [135]:
greedy_iterative_inaccurate_reward = {}
for modification in ["continuous","binary"]:
    if modification == "continuous" and cbm_std_by_concept is not None:
        modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
    elif modification == "binary" and cbm_accuracy_by_concept is not None:
        modified_concept_predictors = [inaccurate_concepts_binary(func,acc) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
    else:
        continue 
    
    subset_concept = [modified_concept_predictors[i] for i in greedy_iterative_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept)
    model = train_ppo_model(env,total_timesteps=10_000,policy="MlpPolicy")
    greedy_iterative_inaccurate_reward[modification] = evaluate_model(environment_string,eval_env,additional_info,model,seed)

In [95]:
lp_inaccurate_reward = {}
for modification in ["continuous","binary"]:
    if modification == "continuous" and cbm_std_by_concept is not None:
        modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
    elif modification == "binary" and cbm_accuracy_by_concept is not None:
        modified_concept_predictors = [inaccurate_concepts_binary(func,acc) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
    else:
        continue 
    
    subset_concept = [modified_concept_predictors[i] for i in lp_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept)
    model = train_ppo_model(env,total_timesteps=10_000,policy="MlpPolicy")
    lp_inaccurate_reward[modification] = evaluate_model(environment_string,eval_env,additional_info,model,seed)

In [ ]:
max_accuracy_selection(pairs,accuracies,direction,target_abstraction_percentage=target_abstraction)

In [67]:
pairs = [[], [0, 1, 2], [0], [2], [], [0], [], [], [0, 2], [1], [0, 1, 2], [2], [], [1], [0], [], [0, 2], [0, 1], [0, 1], [], [], [2], [2], [0], [0, 1], [], [], [0], [0, 1], [], [0, 1], [1], [0], [], [2], [0], [0, 2], [1], [0], [], [0, 1], [1], [2], [0, 1], [0], [0], [0], [0, 2], [0, 2], [0, 2], [0, 1, 2], [0, 1], [], [0, 2], [0, 1], [0, 1, 2], [], [2], [0], [2], [], [0, 1, 2], [0, 2], [0, 1], [1], [0, 1], [1], [0], [], [], [0], [], [2], [2], [], [0], [1], [], [0, 1], [0, 1], [0, 1], [2], [0, 1], [1], [1], [0, 1], [0, 2], [0], [0], [1], [2], [0, 1, 2], [0], [], [1], [], [], [], [1], []]

In [89]:
imperfect_lp_selection_reward = {}
for modification in ["continuous","binary"]:
    if modification == "continuous" and cbm_std_by_concept is not None:
        modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
    elif modification == "binary" and cbm_accuracy_by_concept is not None:
        modified_concept_predictors = [inaccurate_concepts_binary(func,acc) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
    else:
        continue 
    
    if modification == "continuous":
        subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_eval_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,cbm_std_by_concept,direction='min')
    else:
        subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_eval_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,cbm_accuracy_by_concept,direction='max')
    env, eval_env, additional_info = get_environment(environment_string,subset_concept)
    model = train_ppo_model(env,total_timesteps=training_timesteps,policy="MlpPolicy")
    imperfect_lp_selection_reward[modification] = evaluate_model(environment_string,eval_env,additional_info,model,seed)

Pairs [[0, 1, 2], [0, 1, 2], [2], [], [0, 1, 2], [0], [2], [0, 2], [1], [2], [2], [0, 2], [0, 1], [0], [0], [], [0], [0], [0, 1, 2], [1], [1], [1], [], [0, 1], [0, 1], [], [1], [1], [1], [0], [2], [1], [], [0], [2], [], [2], [], [0, 1, 2], [2], [2], [0, 2], [0, 1], [0, 1], [0, 1, 2], [0, 1], [2], [0, 1], [2], [1], [2], [0, 2], [0, 1], [], [1], [0], [0, 1], [2], [], [2], [1], [1], [], [], [], [], [2], [], [0, 1, 2], [0, 1, 2], [2], [], [0, 1, 2], [], [1], [1], [2], [1], [], [0, 1, 2], [], [1], [], [2], [1], [], [0], [0], [0, 1, 2], [0], [0, 1], [0, 2], [2], [0], [2], [2], [], [2], [1], [2]]


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [90]:
perturbed_groundtruth_eval_env = RewardPerturbationWrapper(ground_truth_eval_env)
subset_concept, greedy_perturbed_idx = greedy_selection(perturbed_groundtruth_eval_env,concept_list,num_concepts_selected,groundtruth_model,selection_function)
env, eval_env, additional_info = get_environment(environment_string,subset_concept)
perturbed_model = train_ppo_model(env,total_timesteps=10_000,policy="MlpPolicy")
greedy_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,perturbed_model,seed)


In [139]:
subset_concept, greedy_perturbed_iterative_idx = greedy_iterative_selection(perturbed_groundtruth_eval_env,concept_list,num_concepts_selected,groundtruth_model,selection_function)
env, eval_env, additional_info = get_environment(environment_string,subset_concept)
model = train_ppo_model(env,total_timesteps=training_timesteps,policy="MlpPolicy")
greedy_iterative_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [99]:
subset_concept, lp_perturbed_idx = lp_based_selection(perturbed_groundtruth_eval_env,concept_list,num_concepts_selected,groundtruth_model,selection_function,target_abstraction)
env, eval_env, additional_info = get_environment(environment_string,subset_concept)
model = train_ppo_model(env,total_timesteps=training_timesteps,policy="MlpPolicy")
lp_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)

### Experiment 5

In [104]:
if not isinstance(ground_truth_env, ConceptEnv):
    model = train_two_stage_ppo_model(ground_truth_env,total_timesteps=training_timesteps)
    two_stage_eval = evaluate_model(environment_string,ground_truth_eval_env,additional_info,model,seed)
    per_state_loss = model.per_state_loss
    per_state_acc = None

In [105]:
if not isinstance(ground_truth_env, ConceptEnv):
    ground_truth_concept_env = InfoTransformWrapper(ground_truth_env,[concept_list[i] for i in greedy_idx])
    ground_truth_eval_concept_env = InfoTransformWrapper(ground_truth_eval_env,[concept_list[i] for i in greedy_idx])
    model = train_two_stage_ppo_model(ground_truth_concept_env,total_timesteps=training_timesteps)
    greedy_two_stage_reward = evaluate_model(environment_string,ground_truth_eval_concept_env,additional_info,model,seed)

In [112]:
if not isinstance(ground_truth_env, ConceptEnv):
    ground_truth_concept_env = InfoTransformWrapper(ground_truth_env,[concept_list[i] for i in greedy_iterative_idx])
    ground_truth_eval_concept_env = InfoTransformWrapper(ground_truth_eval_env,[concept_list[i] for i in greedy_iterative_idx])
    model = train_two_stage_ppo_model(ground_truth_concept_env,total_timesteps=training_timesteps)
    greedy_iterative_two_stage_reward = evaluate_model(environment_string,ground_truth_eval_concept_env,additional_info,model,seed)

In [111]:
if not isinstance(ground_truth_env, ConceptEnv):
    ground_truth_concept_env = InfoTransformWrapper(ground_truth_env,[concept_list[i] for i in lp_idx])
    ground_truth_eval_concept_env = InfoTransformWrapper(ground_truth_eval_env,[concept_list[i] for i in lp_idx])
    model = train_two_stage_ppo_model(ground_truth_concept_env,total_timesteps=training_timesteps)
    lp_two_stage_reward = evaluate_model(environment_string,ground_truth_eval_concept_env,additional_info,model,seed)

In [108]:
if not isinstance(ground_truth_env, ConceptEnv) and (cbm_std_by_concept is not None or cbm_accuracy_by_concept is not None):
    if per_state_loss is not None:
        modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,per_state_loss)]
        subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_eval_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,cbm_std_by_concept,direction='min')
    elif per_state_acc is not None:
        modified_concept_predictors = [inaccurate_concepts_binary(func,acc) for func,acc in zip(concept_list,per_state_acc)]
        subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_eval_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,cbm_accuracy_by_concept,direction='max')
    
    ground_truth_concept_env  = InfoTransformWrapper(ground_truth_env,[concept_list[i] for i in imperfect_idx])
    ground_truth_eval_concept_env = InfoTransformWrapper(ground_truth_eval_env,[concept_list[i] for i in imperfect_idx])
    model = train_two_stage_ppo_model(ground_truth_concept_env,total_timesteps=training_timesteps)
    imperfect_two_stage_reward = evaluate_model(environment_string,ground_truth_eval_concept_env,additional_info,model,seed)

In [109]:
if not isinstance(ground_truth_env, ConceptEnv):
    iterative_concepts, iterative_idx = iterative_selection(ground_truth_env,concept_list,groundtruth_model,selection_function,target_abstraction,2,training_timesteps)
    ground_truth_concept_env  = InfoTransformWrapper(ground_truth_env,iterative_concepts)
    ground_truth_eval_concept_env = InfoTransformWrapper(ground_truth_eval_env,iterative_concepts)
    model = train_two_stage_ppo_model(ground_truth_concept_env,total_timesteps=training_timesteps)
    iterative_two_stage_reward = evaluate_model(environment_string,ground_truth_eval_concept_env,additional_info,model,seed)

In [110]:
if not isinstance(ground_truth_env, ConceptEnv):
    bayesian_concepts, bayesian_idx = bayesian_iterative_selection(ground_truth_env,ground_truth_eval_env,environment_string,additional_info,seed,concept_list,2,training_timesteps)
    ground_truth_concept_env  = InfoTransformWrapper(ground_truth_env,bayesian_concepts)
    ground_truth_eval_concept_env = InfoTransformWrapper(ground_truth_eval_env,bayesian_concepts)
    model = train_two_stage_ppo_model(ground_truth_concept_env,total_timesteps=training_timesteps)
    bayesian_two_stage_reward = evaluate_model(environment_string,ground_truth_eval_concept_env,additional_info,model,seed)

## Save Data

In [16]:
save_path = get_save_path(out_folder,save_name)

In [17]:
delete_duplicate_results(out_folder,"",results)

In [18]:
json.dump(results,open('../../results/'+save_path,'w'))